# Data Preprocessing — CICIDS2017

Notebook này chỉ đọc dữ liệu gốc, kiểm tra nhãn, làm sạch và duplicate, audit chất lượng, rồi lưu dataset processed dùng chung. Không chia Train/Test, không fit imputer/scaler/feature selector/SMOTE và không huấn luyện mô hình.

CSV gốc không bị sửa. Dataset đầu ra giữ `SourceFile`, `CaptureDay`, `OriginalLabel`, `BinaryLabel`; các metadata không phải feature mô hình.

## Quy trình trong notebook này

1. Kiểm tra schema, RAM khả dụng và dung lượng đĩa.
2. Audit nhãn gốc bằng cách đọc riêng cột `Label`; chưa map nhãn.
3. Sau khi xác nhận allowlist, xử lý từng file riêng (và dùng chunk nếu một file vượt biên RAM).
4. Đổi Infinity thành missing, loại dòng còn NaN/Infinity và ghi staging riêng theo file.
5. Deduplicate chỉ trong cùng file theo toàn bộ feature + `OriginalLabel`; giữ duplicate liên file và nhóm mâu thuẫn nhãn.
6. Audit giá trị âm nhưng không tự động thay đổi hoặc loại dòng.
7. Hợp nhất staging tuần tự, kiểm tra checksum/số dòng/schema/metadata, rồi mới công bố dataset.

Chạy từng cell theo thứ tự. Sau mỗi bước audit, dừng lại xem output và xác nhận trước khi chạy bước tiếp theo. Cell lỗi hoặc chạy lại không được tự động append vào staging/output có sẵn.

In [8]:
from pathlib import Path
from datetime import datetime
import gc
import gzip
import hashlib
import json
import os
import shutil
import sqlite3
import uuid

import numpy as np
import pandas as pd
import psutil

LABEL_COLUMN = "Label"
METADATA_COLUMNS = ["SourceFile", "CaptureDay", "OriginalLabel", "BinaryLabel"]
DATASET_NAME = "CIC-IDS-2017"
SAMPLE_ROWS = 5_000
DEFAULT_CHUNK_ROWS = 20_000

project_roots = (Path.cwd(), *Path.cwd().parents)
DATA_DIR = next((root / "Dataset" / DATASET_NAME for root in project_roots if (root / "Dataset" / DATASET_NAME).is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Không tìm thấy Dataset/CIC-IDS-2017 từ thư mục hiện tại hoặc các thư mục cha.")

CSV_FILES = sorted(DATA_DIR.glob("*.csv"))
if len(CSV_FILES) != 8:
    raise ValueError(f"Cần đúng 8 file CSV, hiện tìm thấy {len(CSV_FILES)} file.")

OUTPUT_DIR = DATA_DIR.parent / "Processed"
AUDIT_ROOT = OUTPUT_DIR / "audit"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]
RUN_DIR = OUTPUT_DIR / ".staging" / RUN_ID
NONFINITE_STAGE_DIR = RUN_DIR / "nonfinite"
DEDUP_STAGE_DIR = RUN_DIR / "deduplicated"
RUN_AUDIT_DIR = AUDIT_ROOT / RUN_ID

headers = []
for file_path in CSV_FILES:
    header = pd.read_csv(file_path, encoding="latin-1", nrows=0).columns.str.strip().tolist()
    if len(header) != len(set(header)):
        raise ValueError(f"Tên cột bị trùng sau khi strip: {file_path.name}")
    if LABEL_COLUMN not in header:
        raise ValueError(f"Thiếu cột {LABEL_COLUMN}: {file_path.name}")
    headers.append(header)
if any(set(header) != set(headers[0]) for header in headers[1:]):
    raise ValueError("Schema cột của 8 file không đồng nhất.")

INPUT_COLUMNS = headers[0]
FEATURE_COLUMNS = [column for column in INPUT_COLUMNS if column != LABEL_COLUMN]
OUTPUT_COLUMNS = [*FEATURE_COLUMNS, *METADATA_COLUMNS]
SOURCE_BITS = {path.name: 1 << index for index, path in enumerate(CSV_FILES)}
VALID_LABELS = {
    "BENIGN", "Bot", "DDoS", "DoS GoldenEye", "DoS Hulk",
    "DoS Slowhttptest", "DoS slowloris", "FTP-Patator", "Heartbleed",
    "Infiltration", "PortScan", "SSH-Patator", "Web Attack - Brute Force",
    "Web Attack - Sql Injection", "Web Attack - XSS",
    "Web Attack ï¿½ Brute Force", "Web Attack ï¿½ Sql Injection", "Web Attack ï¿½ XSS",
}
LABEL_BITS = {label: 1 << index for index, label in enumerate(sorted(VALID_LABELS))}

def capture_day(file_name):
    name = file_name.lower()
    for day in ("monday", "tuesday", "wednesday", "thursday", "friday"):
        if day in name:
            return day.capitalize()
    return "Unknown"

def binary_counts(frame):
    counts = frame["BinaryLabel"].value_counts().reindex(["Normal", "Attack"], fill_value=0)
    return {"Normal": int(counts["Normal"]), "Attack": int(counts["Attack"])}

print(f"Tìm thấy {len(CSV_FILES)} CSV; feature columns={len(FEATURE_COLUMNS)}; RUN_ID={RUN_ID}")

Tìm thấy 8 CSV; feature columns=78; RUN_ID=20261029_055919_951eee0a


## 1. Preflight RAM và dung lượng đĩa

Ước tính riêng từng CSV bằng mẫu 5.000 dòng. Nếu một file nạp nguyên file có peak ước tính vượt 75% RAM còn trống, các bước xử lý sẽ tự chuyển sang chunks. Cell cũng kiểm tra dung lượng đĩa dự kiến cho staging, SQLite duplicate registry và output gzip. Chỉ chạy cell này trước; chưa đọc toàn bộ flow.

In [9]:
memory_rows = []
total_csv_bytes = sum(path.stat().st_size for path in CSV_FILES)
estimated_pandas_bytes = 0
largest_file_peak = 0
largest_file_bytes_per_row = 0.0

for file_path in CSV_FILES:
    sample = pd.read_csv(file_path, encoding="latin-1", nrows=SAMPLE_ROWS)
    sample_rows = len(sample)
    sample_memory = int(sample.memory_usage(index=True, deep=True).sum())
    with file_path.open("rb") as source:
        header_bytes = len(source.readline())
        sample_data_bytes = 0
        for _ in range(sample_rows):
            line = source.readline()
            if not line:
                break
            sample_data_bytes += len(line)
    sample_data_bytes = max(sample_data_bytes, 1)
    estimated_rows = int(file_path.stat().st_size * sample_rows / sample_data_bytes)
    estimated_bytes = int(sample_memory * estimated_rows / max(sample_rows, 1))
    estimated_peak = int(estimated_bytes * 3.5)
    estimated_pandas_bytes += estimated_bytes
    largest_file_peak = max(largest_file_peak, estimated_peak)
    bytes_per_row = estimated_bytes / max(estimated_rows, 1)
    largest_file_bytes_per_row = max(largest_file_bytes_per_row, bytes_per_row)
    memory_rows.append({
        "File": file_path.name,
        "CSV_MB": file_path.stat().st_size / 1024**2,
        "EstimatedRows": estimated_rows,
        "EstimatedPandas_MB": estimated_bytes / 1024**2,
        "EstimatedPeak_MB": estimated_peak / 1024**2,
    })
    del sample
    gc.collect()

available_ram = psutil.virtual_memory().available
disk_free = shutil.disk_usage(DATA_DIR.parent).free
estimated_disk_required = int(total_csv_bytes * 2.5 + 1024**3)
USE_CHUNKS = largest_file_peak > available_ram * 0.75
if USE_CHUNKS:
    CHUNK_ROWS = max(
        1_000,
        min(DEFAULT_CHUNK_ROWS, int((available_ram * 0.20) / max(largest_file_bytes_per_row * 3.5, 1))),
    )
else:
    CHUNK_ROWS = DEFAULT_CHUNK_ROWS
estimated_chunk_peak = int(largest_file_bytes_per_row * CHUNK_ROWS * 3.5)

memory_preflight = pd.DataFrame(memory_rows)
display(memory_preflight)
print(f"RAM khả dụng: {available_ram / 1024**3:.2f} GiB")
print(f"Peak lớn nhất nếu đọc từng file nguyên vẹn: {largest_file_peak / 1024**3:.2f} GiB")
print(f"Chế độ xử lý: {'chunks' if USE_CHUNKS else 'từng file nguyên vẹn'}; chunk size dự phòng: {CHUNK_ROWS:,}")
print(f"Peak ước tính mỗi chunk: {estimated_chunk_peak / 1024**2:.1f} MiB")
print(f"Đĩa trống tại thư mục Dataset: {disk_free / 1024**3:.2f} GiB; dự kiến cần tối đa khoảng {estimated_disk_required / 1024**3:.2f} GiB")
if disk_free < estimated_disk_required:
    raise OSError("Dung lượng đĩa chưa đủ biên an toàn cho staging, SQLite registry và dataset export.")
if USE_CHUNKS and estimated_chunk_peak > available_ram * 0.50:
    raise MemoryError("Chunk tối thiểu vượt biên RAM an toàn; dừng trước khi xử lý dữ liệu.")
print("Preflight đạt; chưa xử lý toàn bộ flow.")

,File,CSV_MB,EstimatedRows,EstimatedPandas_MB,EstimatedPeak_MB
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,73.551044,242437,158.844399,555.955398
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,73.343437,252054,165.146118,578.011414
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,55.615163,190023,124.502816,435.759856
3,Monday-WorkingHours.pcap_ISCX.csv,168.731611,462869,303.271152,1061.449034
4,Thursday-WorkingHours-Afternoon-Infilteration....,79.252659,267613,175.339681,613.688882
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,49.613250,166014,108.772151,380.702528
6,Tuesday-WorkingHours.pcap_ISCX.csv,128.821368,391104,256.250820,896.877871
7,Wednesday-workingHours.pcap_ISCX.csv,214.735408,630220,412.919305,1445.217566


RAM khả dụng: 6.80 GiB
Peak lớn nhất nếu đọc từng file nguyên vẹn: 1.41 GiB
Chế độ xử lý: từng file nguyên vẹn; chunk size dự phòng: 20,000
Peak ước tính mỗi chunk: 45.9 MiB
Đĩa trống tại thư mục Dataset: 17.58 GiB; dự kiến cần tối đa khoảng 3.06 GiB
Preflight đạt; chưa xử lý toàn bộ flow.


## 2. Audit nhãn gốc trước khi ánh xạ

Đọc riêng cột `Label` theo chunks, không nạp feature vào RAM. So sánh với allowlist CICIDS2017. Ba chuỗi Web Attack có ký tự `ï¿½` được giữ nguyên theo xác nhận của người dùng, không chuẩn hóa. Nếu có nhãn trống hoặc ngoài allowlist, không được chạy bước cleaning/mapping cho đến khi điều tra và thống nhất cách xử lý.

In [10]:
label_rows = []
for file_path in CSV_FILES:
    for label_chunk in pd.read_csv(
        file_path,
        encoding="latin-1",
        usecols=lambda column: column.strip() == LABEL_COLUMN,
        chunksize=100_000,
    ):
        label_chunk.columns = label_chunk.columns.str.strip()
        labels = label_chunk[LABEL_COLUMN].astype("string").str.strip()
        counts = labels.value_counts(dropna=False)
        label_rows.extend(
            {"SourceFile": file_path.name, "OriginalLabel": str(label), "Rows": int(count)}
            for label, count in counts.items()
        )

label_audit_by_file = (
    pd.DataFrame(label_rows)
    .groupby(["SourceFile", "OriginalLabel"], as_index=False, dropna=False)["Rows"]
    .sum()
)
invalid_label_audit = label_audit_by_file[
    label_audit_by_file["OriginalLabel"].isna()
    | ~label_audit_by_file["OriginalLabel"].isin(VALID_LABELS)
].copy()
labels_valid = invalid_label_audit.empty
display(label_audit_by_file)
print("Nhãn thiếu/chưa hợp lệ:")
display(invalid_label_audit)
print(f"Nhãn hợp lệ: {labels_valid}; số nhóm nhãn cần điều tra: {len(invalid_label_audit)}")

,SourceFile,OriginalLabel,Rows
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,BENIGN,97718
1,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,DDoS,128027
2,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,BENIGN,127537
3,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,PortScan,158930
4,Friday-WorkingHours-Morning.pcap_ISCX.csv,BENIGN,189067
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,Bot,1966
6,Monday-WorkingHours.pcap_ISCX.csv,BENIGN,529918
7,Thursday-WorkingHours-Afternoon-Infilteration....,BENIGN,288566
8,Thursday-WorkingHours-Afternoon-Infilteration....,Infiltration,36
9,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,BENIGN,168186


Nhãn thiếu/chưa hợp lệ:


,SourceFile,OriginalLabel,Rows


Nhãn hợp lệ: True; số nhóm nhãn cần điều tra: 0


## 3. Cleaning NaN/Infinity và ghi staging theo file

Chỉ chạy sau khi bạn đã xem bảng nhãn và xác nhận `labels_valid=True`. Mỗi CSV được xử lý riêng; không giữ nhiều file DataFrame cùng lúc. Nếu `USE_CHUNKS=True`, dùng chunk nhỏ hơn. `BENIGN` map thành Normal; các nhãn trong allowlist còn lại map thành Attack. Bảng output ghi rõ dòng non-finite bị loại theo file và nhãn.

In [11]:
if not labels_valid:
    raise ValueError("Có nhãn chưa được xác nhận; dừng trước khi ánh xạ hoặc cleaning.")
if RUN_DIR.exists():
    raise FileExistsError(f"Run staging đã tồn tại: {RUN_DIR}. Không ghi nối; hãy tạo RUN_ID mới sau khi xử lý run cũ.")
if (OUTPUT_DIR / "cicids2017_clean.csv.gz").exists():
    raise FileExistsError("Dataset output đã tồn tại; không ghi đè âm thầm.")

RUN_DIR.mkdir(parents=True)
NONFINITE_STAGE_DIR.mkdir()
RUN_AUDIT_DIR.mkdir(parents=True)

nonfinite_rows = []
numeric_conversion_rows = []
nonfinite_stage_counts = []

for file_path in CSV_FILES:
    stage_path = NONFINITE_STAGE_DIR / f"{file_path.stem}.csv.gz"
    temp_path = stage_path.with_suffix(stage_path.suffix + ".tmp")
    if stage_path.exists() or temp_path.exists():
        raise FileExistsError(f"Staging file đã tồn tại: {stage_path}")
    raw_counts = {"Normal": 0, "Attack": 0}
    kept_counts = {"Normal": 0, "Attack": 0}
    removed_counts = {"Normal": 0, "Attack": 0}
    rows_with_nan = {"Normal": 0, "Attack": 0}
    rows_with_inf = {"Normal": 0, "Attack": 0}
    rows_with_any_nonfinite = {"Normal": 0, "Attack": 0}
    first_output = True

    read_options = {"encoding": "latin-1"}
    if USE_CHUNKS:
        read_options["chunksize"] = CHUNK_ROWS
    chunks = pd.read_csv(file_path, **read_options)
    if isinstance(chunks, pd.DataFrame):
        chunks = iter([chunks])

    with gzip.open(temp_path, "wt", encoding="utf-8", newline="") as output:
        for chunk in chunks:
            chunk.columns = chunk.columns.str.strip()
            if set(chunk.columns) != set(INPUT_COLUMNS):
                raise ValueError(f"Schema thay đổi khi đọc {file_path.name}")
            chunk = chunk[INPUT_COLUMNS].copy()
            chunk["OriginalLabel"] = chunk[LABEL_COLUMN].astype("string").str.strip()
            chunk["BinaryLabel"] = np.where(chunk["OriginalLabel"].eq("BENIGN"), "Normal", "Attack")
            chunk["SourceFile"] = file_path.name
            chunk["CaptureDay"] = capture_day(file_path.name)
            for column in FEATURE_COLUMNS:
                raw_values = chunk[column]
                numeric_values = pd.to_numeric(raw_values, errors="coerce")
                bad_numeric = raw_values.notna() & numeric_values.isna()
                if bad_numeric.any():
                    counts = chunk.loc[bad_numeric].groupby("OriginalLabel", dropna=False).size()
                    numeric_conversion_rows.extend(
                        {"SourceFile": file_path.name, "Feature": column, "OriginalLabel": str(label), "InvalidCells": int(count)}
                        for label, count in counts.items()
                    )
                chunk[column] = numeric_values

            feature_values = chunk[FEATURE_COLUMNS]
            has_nan = feature_values.isna().any(axis=1)
            has_inf = np.isinf(feature_values.to_numpy(dtype="float64", copy=False)).any(axis=1)
            for label in ("Normal", "Attack"):
                label_mask = chunk["BinaryLabel"].eq(label)
                raw_counts[label] += int(label_mask.sum())
                rows_with_nan[label] += int((label_mask & has_nan).sum())
                rows_with_inf[label] += int((label_mask & has_inf).sum())
                rows_with_any_nonfinite[label] += int((label_mask & (has_nan | has_inf)).sum())

            chunk[FEATURE_COLUMNS] = feature_values.replace([np.inf, -np.inf], np.nan)
            invalid_rows = chunk[FEATURE_COLUMNS].isna().any(axis=1)
            for label in ("Normal", "Attack"):
                label_mask = chunk["BinaryLabel"].eq(label)
                removed_counts[label] += int((label_mask & invalid_rows).sum())
            clean_chunk = chunk.loc[~invalid_rows, OUTPUT_COLUMNS]
            counts = clean_chunk["BinaryLabel"].value_counts().reindex(["Normal", "Attack"], fill_value=0)
            for label in ("Normal", "Attack"):
                kept_counts[label] += int(counts[label])
            clean_chunk.to_csv(output, index=False, header=first_output, lineterminator="\n")
            first_output = False

    if first_output:
        pd.DataFrame(columns=OUTPUT_COLUMNS).to_csv(
            temp_path,
            index=False,
            encoding="utf-8",
            compression="gzip",
        )
    os.replace(temp_path, stage_path)
    for label in ("Normal", "Attack"):
        nonfinite_rows.append({
            "SourceFile": file_path.name,
            "BinaryLabel": label,
            "RowsBefore": raw_counts[label],
            "RowsWithNaN": rows_with_nan[label],
            "RowsWithInfinity": rows_with_inf[label],
            "RowsWithAnyNonFinite": rows_with_any_nonfinite[label],
            "RowsRemoved": removed_counts[label],
            "RowsAfter": kept_counts[label],
        })
    nonfinite_stage_counts.append({
        "SourceFile": file_path.name,
        "RowsBefore": sum(raw_counts.values()),
        "NormalBefore": raw_counts["Normal"],
        "AttackBefore": raw_counts["Attack"],
        "RowsRemoved": sum(removed_counts.values()),
        "NormalRemoved": removed_counts["Normal"],
        "AttackRemoved": removed_counts["Attack"],
        "RowsAfter": sum(kept_counts.values()),
        "NormalAfter": kept_counts["Normal"],
        "AttackAfter": kept_counts["Attack"],
    })
    print(f"Hoàn tất non-finite stage: {file_path.name}")
    chunk = None
    clean_chunk = None
    chunks = None
    feature_values = None
    invalid_rows = None
    has_nan = None
    has_inf = None
    raw_values = None
    numeric_values = None
    bad_numeric = None
    label_mask = None
    gc.collect()

nonfinite_audit_by_file = pd.DataFrame(nonfinite_rows)
nonfinite_stage_counts = pd.DataFrame(nonfinite_stage_counts)
numeric_conversion_audit = pd.DataFrame(
    numeric_conversion_rows,
    columns=["SourceFile", "Feature", "OriginalLabel", "InvalidCells"],
)
nonfinite_audit_by_file.to_csv(RUN_AUDIT_DIR / "nonfinite_by_file_and_label.csv", index=False, encoding="utf-8")
nonfinite_stage_counts.to_csv(RUN_AUDIT_DIR / "nonfinite_stage_counts.csv", index=False, encoding="utf-8")
numeric_conversion_audit.to_csv(RUN_AUDIT_DIR / "numeric_conversion_issues.csv", index=False, encoding="utf-8")
display(nonfinite_audit_by_file)
display(nonfinite_stage_counts)
print(f"Staging non-finite theo file: {NONFINITE_STAGE_DIR}")

Hoàn tất non-finite stage: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Hoàn tất non-finite stage: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Hoàn tất non-finite stage: Friday-WorkingHours-Morning.pcap_ISCX.csv
Hoàn tất non-finite stage: Monday-WorkingHours.pcap_ISCX.csv
Hoàn tất non-finite stage: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Hoàn tất non-finite stage: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Hoàn tất non-finite stage: Tuesday-WorkingHours.pcap_ISCX.csv
Hoàn tất non-finite stage: Wednesday-workingHours.pcap_ISCX.csv


,SourceFile,BinaryLabel,RowsBefore,RowsWithNaN,RowsWithInfinity,RowsWithAnyNonFinite,RowsRemoved,RowsAfter
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Normal,97718,4,32,32,32,97686
1,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Attack,128027,0,2,2,2,128025
2,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Normal,127537,15,245,245,245,127292
3,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Attack,158930,0,126,126,126,158804
4,Friday-WorkingHours-Morning.pcap_ISCX.csv,Normal,189067,28,112,112,112,188955
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,Attack,1966,0,10,10,10,1956
6,Monday-WorkingHours.pcap_ISCX.csv,Normal,529918,64,437,437,437,529481
7,Monday-WorkingHours.pcap_ISCX.csv,Attack,0,0,0,0,0,0
8,Thursday-WorkingHours-Afternoon-Infilteration....,Normal,288566,18,207,207,207,288359
9,Thursday-WorkingHours-Afternoon-Infilteration....,Attack,36,0,0,0,0,36


,SourceFile,RowsBefore,NormalBefore,AttackBefore,RowsRemoved,NormalRemoved,AttackRemoved,RowsAfter,NormalAfter,AttackAfter
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,97718,128027,34,32,2,225711,97686,128025
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,127537,158930,371,245,126,286096,127292,158804
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,189067,1966,122,112,10,190911,188955,1956
3,Monday-WorkingHours.pcap_ISCX.csv,529918,529918,0,437,437,0,529481,529481,0
4,Thursday-WorkingHours-Afternoon-Infilteration....,288602,288566,36,207,207,0,288395,288359,36
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,170366,168186,2180,135,135,0,170231,168051,2180
6,Tuesday-WorkingHours.pcap_ISCX.csv,445909,432074,13835,264,261,3,445645,431813,13832
7,Wednesday-workingHours.pcap_ISCX.csv,692703,440031,252672,1297,348,949,691406,439683,251723


Staging non-finite theo file: D:\HUST\GR2\NIDS-ML\Dataset\Processed\.staging\20261029_055919_951eee0a\nonfinite


## 4. Deduplicate trong file và audit duplicate liên file

Đọc staging theo từng file/chunk. Khóa duplicate dùng SHA-256 của vector feature numeric đã chuẩn hóa, kết hợp `SourceFile` và `OriginalLabel` trong registry SQLite. Nhờ vậy, bản trùng cùng file/cùng nhãn chỉ giữ một; cùng feature ở file khác vẫn được giữ. Registry cũng thống kê nhóm mâu thuẫn Normal/Attack và biến thể `OriginalLabel`. Mỗi run dùng database/staging riêng; cell không append vào file output cũ.

In [ ]:
def feature_digests(frame):
    if frame.empty:
        return []
    values = np.ascontiguousarray(frame[FEATURE_COLUMNS].to_numpy(dtype="<f8", copy=True))
    values[values == 0.0] = 0.0
    row_bytes = values.view(np.uint8).reshape(len(values), -1)
    return [hashlib.sha256(memoryview(row)).digest() for row in row_bytes]

if not NONFINITE_STAGE_DIR.exists() or not any(NONFINITE_STAGE_DIR.glob("*.csv.gz")):
    raise FileNotFoundError("Chưa có staging non-finite hoàn tất.")
if DEDUP_STAGE_DIR.exists():
    raise FileExistsError(f"Dedup staging đã tồn tại: {DEDUP_STAGE_DIR}. Không ghi nối.")
db_path = RUN_DIR / "duplicate_registry.sqlite"
if db_path.exists():
    raise FileExistsError(f"Duplicate registry đã tồn tại: {db_path}. Không tái sử dụng âm thầm.")
DEDUP_STAGE_DIR.mkdir()

connection = sqlite3.connect(db_path)
connection.execute("PRAGMA journal_mode=WAL")
connection.execute("PRAGMA synchronous=NORMAL")
connection.execute(
    "CREATE TABLE seen_within (source TEXT, original_label TEXT, digest BLOB, PRIMARY KEY(source, original_label, digest)) WITHOUT ROWID"
)
connection.execute(
    "CREATE TABLE feature_groups (digest BLOB PRIMARY KEY, normal_rows INTEGER NOT NULL, attack_rows INTEGER NOT NULL, source_mask INTEGER NOT NULL, original_mask INTEGER NOT NULL) WITHOUT ROWID"
)
connection.execute(
    "CREATE TEMP TABLE chunk_candidates (source TEXT, original_label TEXT, digest BLOB, PRIMARY KEY(source, original_label, digest)) WITHOUT ROWID"
)

duplicate_counts = []
for file_index, file_path in enumerate(CSV_FILES):
    source_stage = NONFINITE_STAGE_DIR / f"{file_path.stem}.csv.gz"
    target_stage = DEDUP_STAGE_DIR / f"{file_path.stem}.csv.gz"
    temp_target = target_stage.with_suffix(target_stage.suffix + ".tmp")
    if not source_stage.exists():
        raise FileNotFoundError(f"Thiếu staging file: {source_stage}")
    if target_stage.exists() or temp_target.exists():
        raise FileExistsError(f"Dedup output đã tồn tại: {target_stage}")
    input_counts = {"Normal": 0, "Attack": 0}
    removed = {"Normal": 0, "Attack": 0}
    kept = {"Normal": 0, "Attack": 0}
    first_output = True
    source_bit = SOURCE_BITS[file_path.name]

    with gzip.open(temp_target, "wt", encoding="utf-8", newline="") as output:
      for chunk in pd.read_csv(source_stage, compression="gzip", chunksize=CHUNK_ROWS):
        chunk = chunk[OUTPUT_COLUMNS]
        local_duplicate = chunk.duplicated(
            subset=["OriginalLabel", *FEATURE_COLUMNS],
            keep="first",
        )
        for label in ("Normal", "Attack"):
            input_counts[label] += int(chunk["BinaryLabel"].eq(label).sum())
            removed[label] += int((local_duplicate & chunk["BinaryLabel"].eq(label)).sum())
        candidate = chunk.loc[~local_duplicate].copy()
        digests = feature_digests(candidate)
        candidate["__digest"] = digests
        connection.execute("DELETE FROM chunk_candidates")
        connection.executemany(
            "INSERT OR IGNORE INTO chunk_candidates VALUES (?, ?, ?)",
            (
                (file_path.name, str(label), digest)
                for label, digest in zip(candidate["OriginalLabel"], digests)
            ),
        )
        already_seen = {
            (label, digest)
            for label, digest in connection.execute(
                "SELECT c.original_label, c.digest FROM chunk_candidates c JOIN seen_within s USING(source, original_label, digest)"
            )
        }
        keep_mask = np.asarray([
            (str(label), digest) not in already_seen
            for label, digest in zip(candidate["OriginalLabel"], digests)
        ], dtype=bool)
        for label in ("Normal", "Attack"):
            removed[label] += int((~keep_mask & candidate["BinaryLabel"].eq(label).to_numpy()).sum())
        kept_with_digest = candidate.loc[keep_mask].copy()
        kept_digests = kept_with_digest["__digest"].tolist()
        kept_chunk = kept_with_digest.drop(columns="__digest")
        for label in ("Normal", "Attack"):
            kept[label] += int(kept_chunk["BinaryLabel"].eq(label).sum())
        connection.executemany(
            "INSERT OR IGNORE INTO seen_within VALUES (?, ?, ?)",
            (
                (file_path.name, str(label), digest)
                for label, digest in zip(candidate["OriginalLabel"], digests)
            ),
        )

        group_rows = {}
        for digest, binary_label, original_label in zip(
            kept_digests,
            kept_chunk["BinaryLabel"],
            kept_chunk["OriginalLabel"],
        ):
            state = group_rows.setdefault(digest, [0, 0, source_bit, LABEL_BITS[str(original_label)]])
            state[0 if binary_label == "Normal" else 1] += 1
            state[2] |= source_bit
            state[3] |= LABEL_BITS[str(original_label)]
        connection.executemany(
            """INSERT INTO feature_groups VALUES (?, ?, ?, ?, ?)
            ON CONFLICT(digest) DO UPDATE SET
                normal_rows=normal_rows+excluded.normal_rows,
                attack_rows=attack_rows+excluded.attack_rows,
                source_mask=source_mask | excluded.source_mask,
                original_mask=original_mask | excluded.original_mask""",
            ((digest, *state) for digest, state in group_rows.items()),
        )
        connection.commit()
        kept_chunk.to_csv(output, index=False, header=first_output, lineterminator="\n")
        first_output = False

    for label in ("Normal", "Attack"):
        if removed[label] + kept[label] != input_counts[label]:
            raise AssertionError(f"Không bảo toàn số dòng {label} trong {file_path.name}.")
    if first_output:
        pd.DataFrame(columns=OUTPUT_COLUMNS).to_csv(temp_target, index=False, compression="gzip")
    os.replace(temp_target, target_stage)
    duplicate_counts.append({
        "SourceFile": file_path.name,
        "RowsBeforeNormal": input_counts["Normal"],
        "RowsBeforeAttack": input_counts["Attack"],
        "RowsRemovedNormal": removed["Normal"],
        "RowsRemovedAttack": removed["Attack"],
        "RowsKeptNormal": kept["Normal"],
        "RowsKeptAttack": kept["Attack"],
    })
    print(f"Hoàn tất duplicate stage: {file_path.name}")
    chunk = None
    input_counts = None
    candidate = None
    kept_with_digest = None
    kept_chunk = None
    local_duplicate = None
    keep_mask = None
    digests = None
    kept_digests = None
    group_rows = None
    gc.collect()

connection.create_function("bit_count", 1, lambda value: int(value).bit_count())
registry_summary = connection.execute(
    """SELECT
        COUNT(*),
        COALESCE(SUM(CASE WHEN bit_count(source_mask) > 1 THEN 1 ELSE 0 END), 0),
        COALESCE(SUM(CASE WHEN normal_rows > 0 AND attack_rows > 0 THEN 1 ELSE 0 END), 0),
        COALESCE(SUM(CASE WHEN bit_count(original_mask) > 1 THEN 1 ELSE 0 END), 0),
        COALESCE(SUM(CASE WHEN bit_count(source_mask) > 1 THEN normal_rows ELSE 0 END), 0),
        COALESCE(SUM(CASE WHEN bit_count(source_mask) > 1 THEN attack_rows ELSE 0 END), 0),
        COALESCE(SUM(CASE WHEN normal_rows > 0 AND attack_rows > 0 THEN normal_rows ELSE 0 END), 0),
        COALESCE(SUM(CASE WHEN normal_rows > 0 AND attack_rows > 0 THEN attack_rows ELSE 0 END), 0)
    FROM feature_groups"""
).fetchone()
duplicate_summary = pd.DataFrame([{
    "UniqueFeatureGroups": int(registry_summary[0]),
    "CrossFileFeatureGroupsKept": int(registry_summary[1]),
    "ConflictingBinaryLabelGroupsKept": int(registry_summary[2]),
    "MultipleOriginalLabelGroupsKept": int(registry_summary[3]),
    "RowsInCrossFileGroupsNormal": int(registry_summary[4]),
    "RowsInCrossFileGroupsAttack": int(registry_summary[5]),
    "RowsInConflictingGroupsNormal": int(registry_summary[6]),
    "RowsInConflictingGroupsAttack": int(registry_summary[7]),
}])
duplicate_stage_counts = pd.DataFrame(duplicate_counts)
duplicate_impact_by_label = pd.DataFrame([
    {
        "BinaryLabel": label,
        "WithinFileExactDuplicatesRemoved": int(duplicate_stage_counts[f"RowsRemoved{label}"].sum()),
        "RowsKeptAfterDedup": int(duplicate_stage_counts[f"RowsKept{label}"].sum()),
    }
    for label in ("Normal", "Attack")
])
conflict_rows = connection.execute(
    "SELECT digest, normal_rows, attack_rows, source_mask, original_mask FROM feature_groups WHERE normal_rows > 0 AND attack_rows > 0"
).fetchall()
source_names = [path.name for path in CSV_FILES]
original_labels = sorted(VALID_LABELS)
conflicting_feature_groups = pd.DataFrame([
    {
        "FeatureGroupID": digest.hex(),
        "NormalRows": int(normal_rows),
        "AttackRows": int(attack_rows),
        "SourceFiles": " | ".join(name for index, name in enumerate(source_names) if source_mask & (1 << index)),
        "CaptureDays": " | ".join(sorted({capture_day(name) for index, name in enumerate(source_names) if source_mask & (1 << index)})),
        "OriginalLabels": " | ".join(label for index, label in enumerate(original_labels) if original_mask & LABEL_BITS[label]),
    }
    for digest, normal_rows, attack_rows, source_mask, original_mask in conflict_rows
])
duplicate_summary.to_csv(RUN_AUDIT_DIR / "duplicate_group_summary.csv", index=False, encoding="utf-8")
duplicate_stage_counts.to_csv(RUN_AUDIT_DIR / "duplicate_counts_by_file.csv", index=False, encoding="utf-8")
duplicate_impact_by_label.to_csv(RUN_AUDIT_DIR / "duplicate_impact_by_label.csv", index=False, encoding="utf-8")
conflicting_feature_groups.to_csv(RUN_AUDIT_DIR / "conflicting_feature_groups.csv", index=False, encoding="utf-8")
connection.close()
display(duplicate_summary)
display(duplicate_stage_counts)
display(duplicate_impact_by_label)
display(conflicting_feature_groups.head(20))
print(f"Deduplicated staging: {DEDUP_STAGE_DIR}")

## 5. Đối soát audit duplicate

Bước này chỉ đọc cột `BinaryLabel` từ non-finite staging và deduplicated staging theo chunks; không đọc lại CSV gốc, không hash/lọc lại feature và không thay đổi các file staging. Cell kiểm tra counts với audit non-finite và kept counts của duplicate stage, tính lại `RowsRemoved` theo từng file/nhãn, rồi xác nhận `before = removed + kept`. Chỉ sau khi tất cả kiểm tra đạt mới cập nhật hai file audit bị sai bằng kết quả đã đối soát.

In [13]:
if not NONFINITE_STAGE_DIR.exists() or not DEDUP_STAGE_DIR.exists():
    raise FileNotFoundError("Missing non-finite or deduplicated staging directory.")
required_audits = {
    "nonfinite_stage_counts.csv": RUN_AUDIT_DIR / "nonfinite_stage_counts.csv",
    "duplicate_counts_by_file.csv": RUN_AUDIT_DIR / "duplicate_counts_by_file.csv",
    "duplicate_group_summary.csv": RUN_AUDIT_DIR / "duplicate_group_summary.csv",
    "conflicting_feature_groups.csv": RUN_AUDIT_DIR / "conflicting_feature_groups.csv",
}
for audit_path in required_audits.values():
    if not audit_path.exists():
        raise FileNotFoundError(f"Missing required audit file: {audit_path}")

source_audit = pd.read_csv(required_audits["nonfinite_stage_counts.csv"])
initial_duplicate_audit = pd.read_csv(required_audits["duplicate_counts_by_file.csv"])

def count_binary_labels(stage_path):
    counts = {"Normal": 0, "Attack": 0}
    for label_chunk in pd.read_csv(
        stage_path,
        compression="gzip",
        usecols=["BinaryLabel"],
        chunksize=CHUNK_ROWS,
    ):
        labels = label_chunk["BinaryLabel"]
        if not labels.isin(counts).all():
            raise ValueError(f"Unknown BinaryLabel in staging file: {stage_path.name}")
        observed = labels.value_counts()
        for label in counts:
            counts[label] += int(observed.get(label, 0))
    return counts

reconciled_rows = []
for file_path in CSV_FILES:
    source_stage = NONFINITE_STAGE_DIR / f"{file_path.stem}.csv.gz"
    dedup_stage = DEDUP_STAGE_DIR / f"{file_path.stem}.csv.gz"
    if not source_stage.exists() or not dedup_stage.exists():
        raise FileNotFoundError(f"Missing stage for {file_path.name}")

    source_counts = count_binary_labels(source_stage)
    kept_counts = count_binary_labels(dedup_stage)
    source_reference = source_audit.loc[source_audit["SourceFile"].eq(file_path.name)]
    duplicate_reference = initial_duplicate_audit.loc[
        initial_duplicate_audit["SourceFile"].eq(file_path.name)
    ]
    if len(source_reference) != 1 or len(duplicate_reference) != 1:
        raise ValueError(f"Expected exactly one audit row for {file_path.name}")
    source_reference = source_reference.iloc[0]
    duplicate_reference = duplicate_reference.iloc[0]

    expected_source = {
        "Normal": int(source_reference["NormalAfter"]),
        "Attack": int(source_reference["AttackAfter"]),
    }
    expected_kept = {
        "Normal": int(duplicate_reference["RowsKeptNormal"]),
        "Attack": int(duplicate_reference["RowsKeptAttack"]),
    }
    if source_counts != expected_source:
        raise ValueError(f"Non-finite staging count mismatch for {file_path.name}")
    if kept_counts != expected_kept:
        raise ValueError(f"Deduplicated staging count mismatch for {file_path.name}")

    removed_counts = {
        label: source_counts[label] - kept_counts[label]
        for label in ("Normal", "Attack")
    }
    if any(value < 0 for value in removed_counts.values()):
        raise ValueError(f"Deduplicated stage has more rows than source for {file_path.name}")
    for label in ("Normal", "Attack"):
        if source_counts[label] != removed_counts[label] + kept_counts[label]:
            raise AssertionError(f"Row conservation failed for {file_path.name}/{label}")

    reconciled_rows.append({
        "SourceFile": file_path.name,
        "RowsBeforeNormal": source_counts["Normal"],
        "RowsBeforeAttack": source_counts["Attack"],
        "RowsRemovedNormal": removed_counts["Normal"],
        "RowsRemovedAttack": removed_counts["Attack"],
        "RowsKeptNormal": kept_counts["Normal"],
        "RowsKeptAttack": kept_counts["Attack"],
    })

if len(reconciled_rows) != len(CSV_FILES):
    raise AssertionError("Not all source files were reconciled.")
duplicate_stage_counts = pd.DataFrame(reconciled_rows)
duplicate_impact_by_label = pd.DataFrame([
    {
        "BinaryLabel": label,
        "WithinFileExactDuplicatesRemoved": int(duplicate_stage_counts[f"RowsRemoved{label}"].sum()),
        "RowsKeptAfterDedup": int(duplicate_stage_counts[f"RowsKept{label}"].sum()),
    }
    for label in ("Normal", "Attack")
])

corrected_counts_path = RUN_AUDIT_DIR / "duplicate_counts_by_file.csv"
corrected_impact_path = RUN_AUDIT_DIR / "duplicate_impact_by_label.csv"
counts_temp = corrected_counts_path.with_suffix(corrected_counts_path.suffix + ".reconciled.tmp")
impact_temp = corrected_impact_path.with_suffix(corrected_impact_path.suffix + ".reconciled.tmp")
if counts_temp.exists() or impact_temp.exists():
    raise FileExistsError("A reconciliation temporary audit file already exists.")
duplicate_stage_counts.to_csv(counts_temp, index=False, encoding="utf-8")
duplicate_impact_by_label.to_csv(impact_temp, index=False, encoding="utf-8")
os.replace(counts_temp, corrected_counts_path)
os.replace(impact_temp, corrected_impact_path)

duplicate_summary = pd.read_csv(required_audits["duplicate_group_summary.csv"])
conflicting_feature_groups = pd.read_csv(required_audits["conflicting_feature_groups.csv"])
display(duplicate_stage_counts)
display(duplicate_impact_by_label)
display(duplicate_summary)
display(conflicting_feature_groups.head(20))
print("Duplicate removal counts reconciled; original raw CSVs and staging files were not modified.")

,SourceFile,RowsBeforeNormal,RowsBeforeAttack,RowsRemovedNormal,RowsRemovedAttack,RowsKeptNormal,RowsKeptAttack
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,97686,128025,2618,11,95068,128014
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,127292,158804,4209,68110,123083,90694
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,188955,1956,6859,8,182096,1948
3,Monday-WorkingHours.pcap_ISCX.csv,529481,0,26831,0,502650,0
4,Thursday-WorkingHours-Afternoon-Infilteration....,288359,36,35605,0,252754,36
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,168051,2180,6015,37,162036,2143
6,Tuesday-WorkingHours.pcap_ISCX.csv,431813,13832,19337,4682,412476,9150
7,Wednesday-workingHours.pcap_ISCX.csv,439683,251723,22947,57967,416736,193756


,BinaryLabel,WithinFileExactDuplicatesRemoved,RowsKeptAfterDedup
0,Normal,124421,2146899
1,Attack,130815,425741


,UniqueFeatureGroups,CrossFileFeatureGroupsKept,ConflictingBinaryLabelGroupsKept,MultipleOriginalLabelGroupsKept,RowsInCrossFileGroupsNormal,RowsInCrossFileGroupsAttack,RowsInConflictingGroupsNormal,RowsInConflictingGroupsAttack
0,2520101,34175,697,697,86017,663,715,697


,FeatureGroupID,NormalRows,AttackRows,SourceFiles,CaptureDays,OriginalLabels
0,004e7a879dd3e45a8e6a26f935c54f13e1452a1dd46056...,1,1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Friday | Thursday,BENIGN | PortScan
1,00d34f6410da3e652fe04cc479c4b3a44157cdfd9aa6c7...,1,1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Friday | Thursday,BENIGN | PortScan
2,013776f6bfa565e03253de2b781884852430be592f28a1...,1,1,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.c...,Friday | Thursday,BENIGN | DDoS
3,019abd80f9cf4234b47fba56e099a4b8275cb2cb310f29...,1,1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Friday | Thursday,BENIGN | PortScan
4,01d2831cb37ed01256061ba64216e056fa8784656c3345...,1,1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Friday | Thursday,BENIGN | PortScan
5,01ded0df8c2838e5ef04e46d6dc080cae0091771bb2976...,2,1,Thursday-WorkingHours-Afternoon-Infilteration....,Thursday | Wednesday,BENIGN | DoS Hulk
6,041f3f891c075e75ea60db5c4f4a3b6263c797f7d2c945...,1,1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Friday | Thursday,BENIGN | PortScan
7,042510f846b0cf44969602c5bedcbdde239aa445d21532...,1,1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Friday | Thursday,BENIGN | PortScan
8,044cfa29ae9d9c1169af2f659835157c00a6aeb2b50434...,1,1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Friday | Thursday,BENIGN | PortScan
9,046fe8fe015650a1af61e736a8fffd5600885a90acd2fd...,1,1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Friday | Thursday,BENIGN | PortScan


Duplicate removal counts reconciled; original raw CSVs and staging files were not modified.


## 6. Audit giá trị âm — không tự động xóa

Đọc deduplicated staging theo chunks. Chỉ thống kê số dòng và số ô bị gắn cờ; không sửa giá trị hoặc loại dòng. Kết quả phải cho `RowsRemoved=0` và label counts trước/sau giống nhau.

In [14]:
negative_rows = []
negative_cells = []
negative_stage_counts = []

for file_path in CSV_FILES:
    stage_path = DEDUP_STAGE_DIR / f"{file_path.stem}.csv.gz"
    file_counts = {"Normal": 0, "Attack": 0}
    flagged = {
        "Init_Win_bytes_-1": {"Normal": 0, "Attack": 0},
        "Negative_rate": {"Normal": 0, "Attack": 0},
        "Very_negative_header": {"Normal": 0, "Attack": 0},
        "Negative_Flow_IAT_Min": {"Normal": 0, "Attack": 0},
    }
    cell_counts = {column: 0 for column in FEATURE_COLUMNS}
    for chunk in pd.read_csv(stage_path, compression="gzip", chunksize=CHUNK_ROWS):
        for label in ("Normal", "Attack"):
            label_mask = chunk["BinaryLabel"].eq(label)
            file_counts[label] += int(label_mask.sum())
            flag_masks = {
                "Init_Win_bytes_-1": (
                    chunk.get("Init_Win_bytes_forward", pd.Series(False, index=chunk.index)).eq(-1)
                    | chunk.get("Init_Win_bytes_backward", pd.Series(False, index=chunk.index)).eq(-1)
                ),
                "Negative_rate": (
                    chunk.get("Flow Packets/s", pd.Series(np.nan, index=chunk.index)).lt(0)
                    | chunk.get("Flow Bytes/s", pd.Series(np.nan, index=chunk.index)).lt(0)
                ),
                "Very_negative_header": chunk.filter(regex="Header Length").lt(-1_000_000).any(axis=1),
                "Negative_Flow_IAT_Min": chunk.get("Flow IAT Min", pd.Series(np.nan, index=chunk.index)).lt(0),
            }
            for issue, mask in flag_masks.items():
                flagged[issue][label] += int((label_mask & mask).sum())
        for column in FEATURE_COLUMNS:
            cell_counts[column] += int(chunk[column].lt(0).sum())
    for issue, counts in flagged.items():
        negative_rows.append({
            "SourceFile": file_path.name,
            "IssueGroup": issue,
            "RowsNormal": counts["Normal"],
            "RowsAttack": counts["Attack"],
            "RowsRemoved": 0,
        })
    negative_cells.extend(
        {"SourceFile": file_path.name, "Feature": column, "NegativeCells": count}
        for column, count in cell_counts.items()
    )
    negative_stage_counts.append({
        "SourceFile": file_path.name,
        "NormalAfterDedup": file_counts["Normal"],
        "AttackAfterDedup": file_counts["Attack"],
        "RowsRemovedByNegativeAudit": 0,
    })
    chunk = None
    gc.collect()

negative_value_audit = pd.DataFrame(negative_rows)
negative_cell_audit = pd.DataFrame(negative_cells)
negative_stage_counts = pd.DataFrame(negative_stage_counts)
negative_value_audit.to_csv(RUN_AUDIT_DIR / "negative_value_audit.csv", index=False, encoding="utf-8")
negative_cell_audit.to_csv(RUN_AUDIT_DIR / "negative_cells_by_feature.csv", index=False, encoding="utf-8")
negative_stage_counts.to_csv(RUN_AUDIT_DIR / "negative_stage_counts.csv", index=False, encoding="utf-8")
display(negative_value_audit)
display(negative_stage_counts)
print("Giá trị âm được giữ nguyên; không có dòng nào bị loại.")

,SourceFile,IssueGroup,RowsNormal,RowsAttack,RowsRemoved
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Init_Win_bytes_-1,39192,46524,0
1,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Negative_rate,2,0,0
2,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Very_negative_header,0,0,0
3,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Negative_Flow_IAT_Min,89,19,0
4,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Init_Win_bytes_-1,72998,64,0
5,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Negative_rate,35,0,0
6,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Very_negative_header,0,0,0
7,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Negative_Flow_IAT_Min,175,0,0
8,Friday-WorkingHours-Morning.pcap_ISCX.csv,Init_Win_bytes_-1,111909,0,0
9,Friday-WorkingHours-Morning.pcap_ISCX.csv,Negative_rate,9,0,0


,SourceFile,NormalAfterDedup,AttackAfterDedup,RowsRemovedByNegativeAudit
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,95068,128014,0
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,123083,90694,0
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,182096,1948,0
3,Monday-WorkingHours.pcap_ISCX.csv,502650,0,0
4,Thursday-WorkingHours-Afternoon-Infilteration....,252754,36,0
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,162036,2143,0
6,Tuesday-WorkingHours.pcap_ISCX.csv,412476,9150,0
7,Wednesday-workingHours.pcap_ISCX.csv,416736,193756,0


Giá trị âm được giữ nguyên; không có dòng nào bị loại.


## 7. Hợp nhất tuần tự và xác minh dataset

Chỉ chạy sau khi đã duyệt audit cleaning. Cell sao chép tuần tự payload từng gzip staging vào một file tạm, giữ đúng một header; không concat DataFrame. Trước khi công bố file, đối chiếu SHA-256 payload, schema, tổng dòng, phân bố theo `SourceFile`/nhãn, metadata và non-finite bằng cách đọc output theo chunks. Nếu output đích đã tồn tại, cell dừng thay vì append/ghi đè.

In [15]:
final_path = OUTPUT_DIR / "cicids2017_clean.csv.gz"
if final_path.exists():
    raise FileExistsError(f"Output đã tồn tại; không ghi nối/ghi đè: {final_path}")
if not DEDUP_STAGE_DIR.exists() or not (DEDUP_STAGE_DIR / f"{CSV_FILES[0].stem}.csv.gz").exists():
    raise FileNotFoundError("Chưa có deduplicated staging hoàn chỉnh.")
temp_final = RUN_DIR / "cicids2017_clean.csv.gz.tmp"
if temp_final.exists():
    raise FileExistsError(f"Temporary output đã tồn tại: {temp_final}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

expected_hash = hashlib.sha256()
with gzip.open(temp_final, "wb") as raw_output:
    for index, file_path in enumerate(CSV_FILES):
        stage_path = DEDUP_STAGE_DIR / f"{file_path.stem}.csv.gz"
        if not stage_path.exists():
            raise FileNotFoundError(f"Thiếu deduplicated staging: {stage_path}")
        with gzip.open(stage_path, "rb") as source:
            header = source.readline()
            if index == 0:
                raw_output.write(header)
            elif header != expected_header:
                raise ValueError(f"Header staging không đồng nhất: {file_path.name}")
            if index == 0:
                expected_header = header
            while True:
                block = source.read(1024 * 1024)
                if not block:
                    break
                raw_output.write(block)
                expected_hash.update(block)

actual_hash = hashlib.sha256()
with gzip.open(temp_final, "rb") as merged:
    merged.readline()
    while True:
        block = merged.read(1024 * 1024)
        if not block:
            break
        actual_hash.update(block)
if actual_hash.digest() != expected_hash.digest():
    raise IOError("Checksum payload sau merge không khớp staging; không công bố output.")

expected_counts = (
    duplicate_stage_counts.groupby("SourceFile")[["RowsKeptNormal", "RowsKeptAttack"]]
    .sum()
)
actual_counts = {}
actual_rows = 0
actual_schema = None
for chunk in pd.read_csv(temp_final, compression="gzip", chunksize=CHUNK_ROWS):
    if actual_schema is None:
        actual_schema = chunk.columns.tolist()
    elif chunk.columns.tolist() != actual_schema:
        raise ValueError("Schema thay đổi khi đọc output theo chunks.")
    if actual_schema != OUTPUT_COLUMNS:
        raise ValueError("Schema output không khớp schema dự kiến.")
    if not chunk["BinaryLabel"].isin(["Normal", "Attack"]).all():
        raise ValueError("Output có BinaryLabel không hợp lệ.")
    if not chunk["OriginalLabel"].isin(VALID_LABELS).all():
        raise ValueError("Output có OriginalLabel không hợp lệ.")
    if chunk[METADATA_COLUMNS].isna().any().any():
        raise ValueError("Output có metadata thiếu.")
    if not np.isfinite(chunk[FEATURE_COLUMNS].to_numpy(dtype="float64", copy=False)).all():
        raise ValueError("Output còn NaN/Infinity trong feature.")
    actual_rows += len(chunk)
    grouped = chunk.groupby(["SourceFile", "BinaryLabel"]).size()
    for (source, label), count in grouped.items():
        actual_counts[(source, label)] = actual_counts.get((source, label), 0) + int(count)

expected_rows = int(expected_counts.to_numpy().sum())
if actual_rows != expected_rows:
    raise ValueError(f"Số dòng output {actual_rows:,} khác audit staging {expected_rows:,}.")
for source, row in expected_counts.iterrows():
    if actual_counts.get((source, "Normal"), 0) != int(row["RowsKeptNormal"]):
        raise ValueError(f"Số Normal không khớp audit: {source}")
    if actual_counts.get((source, "Attack"), 0) != int(row["RowsKeptAttack"]):
        raise ValueError(f"Số Attack không khớp audit: {source}")

merge_validation = pd.DataFrame([{
    "Rows": actual_rows,
    "SchemaColumns": len(actual_schema),
    "ChecksumMatchesStaging": True,
    "MetadataComplete": True,
    "NoNonFiniteFeatures": True,
    "OutputTemporary": str(temp_final),
}])
merge_validation.to_csv(RUN_AUDIT_DIR / "merge_validation.csv", index=False, encoding="utf-8")
os.rename(temp_final, final_path)
print(f"Dataset verified and published: {final_path}")
print(f"Rows: {actual_rows:,}; columns: {len(actual_schema)}; SHA-256 staging payload matched.")
print(f"Audit directory: {RUN_AUDIT_DIR}")
display(merge_validation)

Dataset verified and published: D:\HUST\GR2\NIDS-ML\Dataset\Processed\cicids2017_clean.csv.gz
Rows: 2,572,640; columns: 82; SHA-256 staging payload matched.
Audit directory: D:\HUST\GR2\NIDS-ML\Dataset\Processed\audit\20261029_055919_951eee0a


,Rows,SchemaColumns,ChecksumMatchesStaging,MetadataComplete,NoNonFiniteFeatures,OutputTemporary
0,2572640,82,True,True,True,D:\HUST\GR2\NIDS-ML\Dat...
